<a href="https://colab.research.google.com/github/JulTob/Optimization/blob/main/Practicas/2026_2027_Enunciado_Pr%C3%A1ctica_1_e.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración
!pip install cplex
!pip install docplex
from docplex.mp.model import Model
from docplex.mp.environment import Environment
env = Environment()
env.print_information()

import numpy as np
import matplotlib.pyplot as plt

* system is: Linux 64bit
* Python version 3.13.16, located at: /usr/bin/python3
* docplex is present, version is 2.32.264
* CPLEX library is present, version is 22.2.0.0, located at: /usr/local/lib/python3.13/dist-packages
* pandas is present, version is 2.2.3


# <font color="steelblue">**EJERCICIOS  PRÁCTICA 1:**</font>

# <font color="steelblue">**Problema Planificación Agrícola**</font>


**Resolver el siguiente  problema de optimización**    

A continuación se describe un estudio pormenorizado del
ejemplo del granjero, tomado en su enunciado inicial del libro
de *Birge y Louveaux (1997)*.


Consideremos un granjero que quiere cultivar **trigo**, **maíz** y
**remolacha azucarera** en **500 acres de tierra**.

Durante el invierno quiere decidir cuánta tierra destinar a cada
tipo de cosecha. El granjero sabe que se necesitan **al menos**
**200 toneladas de trigo** y **240 de maíz** para la comida del
ganado. Estas cantidades pueden obtenerse de la granja o
comprarse en un almacén y cualquier exceso de producción se
venderá.

Los **precios de venta** son de **170** y **150** euros por tonelada de
**trigo y de maíz**; mientras que los **precios de compra** son un
**40% más**, debido a los márgenes del almacén y los costes de
transporte.
    
Otro cultivo beneficioso es la remolacha azucarera que se
vende a **26 euros** por tonelada. Sin embargo, la  **Comisión
Europea impone una cuota** a la producción de remolacha.

Cualquier **exceso** de la **cuota** se vende a **10 euros** por tonelada.

La **cuota** del granjero para este año es de **6000** toneladas.

Por la experiencia de años anteriores, el granjero sabe que el **rendimiento medio** en sus tierras es de **2.5, 3 y 20** toneladas,
para el cultivo del **trigo, maíz y remolacha**, respectivamente.

Los **costes** de plantación son **150, 230 y 260** euros/acre de
**trigo, maíz y remolacha**, respectivamente.

La pregunta es: **¿Cuál será la distribución óptima de los
cultivos que minimiza los gastos del granjero?**
    
    


![Datos_Planificacion_Agricola](https://drive.google.com/thumbnail?id=16iXoIdDf9nYunUvSTvusYMwMntftpcOq&sz=h350)


  <span style="color:brown">

  #### <font color='steelblue'> **Modelo Determinista** </font>
    
$Z_{PL} = \min 150 x_1 + 230 x_ 2 + 260 x_3 + 238 y_1 + 210 y_2
−170 w_1 − 150 w_2 − 36 w_3 − 10 w_ 4$
    
$\quad s.a$
    
$\qquad   x_1 + x_2 + x_3 ≤ 500$
    
$\qquad  2.5 x_1 + y_1 − w_1 ≥ 200  \quad \leftrightarrow \quad −2,5 x_1 − y_1 + w_1 ≤ −200$
    
$\qquad 3x2 + y2 − w2 ≥ 240 \quad\leftrightarrow \quad −3x2 − y2 + w2 ≤ −240 $

$\qquad  w_3 + w_4 ≤ 20 x_3 \quad\leftrightarrow \quad w_3 + w_4 − 20 x_3 ≤ 0$

$\qquad w_3 ≤ 6000$

$\quad x_i ≥ 0, ∀i = 1,2,3$
    
$\quad y_i ≥ 0, ∀i = 1, 2$

$\quad w_i ≥ 0, ∀i = 1, ..., 4$    

In [8]:
#@title <font color="steelblue" size="+1"> Modelo CPLEX

Rendimientos=[2.5, 3, 20]
Coste = [150,230,260]
Precio_Venta = [170,150,36,10 ]
Precio_Compra = [238,210]
Rendimiento_Min= [200,240]
Total_acres=500

N_x= range(len(Rendimientos))
N_y= range(len(Precio_Compra))
N_w=range(len(Precio_Venta))

#Modelo
mdl = Model()




In [21]:
def resolver_problema_agricola(
        Rendimientos,
        Coste,
        Precio_Venta,
        Precio_Compra,
        Rendimiento_Min,
        Total_acres,
        uds=["Trigo[Tm]", "Maíz[Tm]", "Remolacha[Tm]"],
        titulo="Problema Agrícola"):
    from docplex.mp.model import Model

    mdl = Model(name=titulo)

    # Índices
    N_x = range(len(Rendimientos))
    N_y = range(len(Precio_Compra))
    N_w = range(len(Precio_Venta))

    # Variables de decisión
    x = mdl.continuous_var_list(N_x, lb=0, name='x')  # Acres asignados
    y = mdl.continuous_var_list(N_y, lb=0, name='y')  # Toneladas compradas
    w = mdl.continuous_var_list(N_w, lb=0, name='w')  # Toneladas vendidas

    # Función Objetivo
    costes_plantacion = mdl.sum(Coste[i] * x[i] for i in N_x)
    costes_compra = mdl.sum(Precio_Compra[i] * y[i] for i in N_y)
    ingresos_venta = mdl.sum(Precio_Venta[i] * w[i] for i in N_w)

    mdl.minimize(costes_plantacion + costes_compra - ingresos_venta)

    # Restricciones
    mdl.add_constraint(mdl.sum(x[i] for i in N_x) <= Total_acres, ctname='Tierra')
    mdl.add_constraint(Rendimientos[0] * x[0] + y[0] - w[0] >= Rendimiento_Min[0], ctname='Req_Trigo')
    mdl.add_constraint(Rendimientos[1] * x[1] + y[1] - w[1] >= Rendimiento_Min[1], ctname='Req_Maiz')
    mdl.add_constraint(w[2] + w[3] <= Rendimientos[2] * x[2], ctname='Prod_Remolacha')
    mdl.add_constraint(w[2] <= 6000, ctname='Cuota_Remolacha')

    # Resolver
    sol = mdl.solve()

    if sol:
        print(f"=== {titulo} ===")
        print(f"Estado de la solución: {mdl.get_solve_status()}")
        print(f"Coste Neto Mínimo: {sol.objective_value:.2f} €")
        print("\nDistribución de cultivos (Acres a plantar):")
        for i in N_x:
            nombre = uds[i] if i < len(uds) else f"Cultivo {i+1}"
            print(f"  - {nombre}: {x[i].solution_value:.2f} acres (Producción: {x[i].solution_value * Rendimientos[i]:.2f})")

        print("\nCompras realizadas:")
        for i in N_y:
            nombre = uds[i] if i < len(uds) else f"Cultivo {i+1}"
            print(f"  - {nombre} comprado: {y[i].solution_value:.2f}")

        print("\nVentas realizadas:")
        for i in N_y:
            nombre = uds[i] if i < len(uds) else f"Cultivo {i+1}"
            print(f"  - {nombre} vendido: {w[i].solution_value:.2f}")
        if len(w) > 2:
            precio_normal = Precio_Venta[2]
            print(f"  - {uds[2]} (precio normal {precio_normal:.2f} €): {w[2].solution_value:.2f}")
        if len(w) > 3:
            precio_exceso = Precio_Venta[3]
            print(f"  - {uds[2]} (precio exceso {precio_exceso:.2f} €): {w[3].solution_value:.2f}")
        print("=" * (len(titulo) + 8) + "\n")
    else:
        print(f"No se encontró solución para: {titulo}")

    return mdl, sol

# Resolver el caso base determinista utilizando las variables cargadas
mdl_base, sol_base = resolver_problema_agricola(
    Rendimientos,
    Coste,
    Precio_Venta,
    Precio_Compra,
    Rendimiento_Min,
    Total_acres,
    uds=["Trigo[Tm]", "Maíz[Tm]", "Remolacha[Tm]"],
    titulo="Caso Determinista Base"
)

=== Caso Determinista Base ===
Estado de la solución: JobSolveStatus.OPTIMAL_SOLUTION
Coste Neto Mínimo: -118600.00 €

Distribución de cultivos (Acres a plantar):
  - Trigo[Tm]: 120.00 acres (Producción: 300.00)
  - Maíz[Tm]: 80.00 acres (Producción: 240.00)
  - Remolacha[Tm]: 300.00 acres (Producción: 6000.00)

Compras realizadas:
  - Trigo[Tm] comprado: 0.00
  - Maíz[Tm] comprado: 0.00

Ventas realizadas:
  - Trigo[Tm] vendido: 100.00
  - Maíz[Tm] vendido: 0.00
  - Remolacha[Tm] (precio normal 36.00 €): 6000.00
  - Remolacha[Tm] (precio exceso 10.00 €): 0.00




 Ahora bien, por cambios meteorológicos, los rendimientos pueden variar un 20% arriba o abajo.

In [22]:
#@title <font color="steelblue" size="+1"> rendimientos +20%
### Caso rendimientos 20% superiores

# Calculamos los rendimientos incrementados en un 20%
Rendimientos_mas_20 = [r * 1.2 for r in Rendimientos]

# Resolvemos usando la función generalizada
mdl_mas_20, sol_mas_20 = resolver_problema_agricola(
    Rendimientos=Rendimientos_mas_20,
    Coste=Coste,
    Precio_Venta=Precio_Venta,
    Precio_Compra=Precio_Compra,
    Rendimiento_Min=Rendimiento_Min,
    Total_acres=Total_acres,
    uds=["Trigo[Tm]", "Maíz[Tm]", "Remolacha[Tm]"],
    titulo="Caso Rendimientos +20%"
)

=== Caso Rendimientos +20% ===
Estado de la solución: JobSolveStatus.OPTIMAL_SOLUTION
Coste Neto Mínimo: -167666.67 €

Distribución de cultivos (Acres a plantar):
  - Trigo[Tm]: 183.33 acres (Producción: 550.00)
  - Maíz[Tm]: 66.67 acres (Producción: 240.00)
  - Remolacha[Tm]: 250.00 acres (Producción: 6000.00)

Compras realizadas:
  - Trigo[Tm] comprado: 0.00
  - Maíz[Tm] comprado: 0.00

Ventas realizadas:
  - Trigo[Tm] vendido: 350.00
  - Maíz[Tm] vendido: 0.00
  - Remolacha[Tm] (precio normal 36.00 €): 6000.00
  - Remolacha[Tm] (precio exceso 10.00 €): 0.00



In [19]:
#@title <font color="steelblue" size="+1"> rendimientos -20%
### Caso rendimientos 20% inferiores

# Calculamos los rendimientos reducidos en un 20%
Rendimientos_menos_20 = [r * 0.8 for r in Rendimientos]

# Resolvemos usando la función generalizada
mdl_menos_20, sol_menos_20 = resolver_problema_agricola(
    Rendimientos=Rendimientos_menos_20,
    Coste=Coste,
    Precio_Venta=Precio_Venta,
    Precio_Compra=Precio_Compra,
    Rendimiento_Min=Rendimiento_Min,
    Total_acres=Total_acres,
    uds=["Trigo[Tm]", "Maíz[Tm]", "Remolacha[Tm]"],
    titulo="Caso Rendimientos -20%"
)

=== Caso Rendimientos -20% ===
Estado de la solución: JobSolveStatus.OPTIMAL_SOLUTION
Coste Neto Mínimo: -59950.00 €

Distribución de cultivos (Acres a plantar):
  - Trigo[Tm]: 100.00 acres (Producción: 200.00 t)
  - Maíz[Tm]: 25.00 acres (Producción: 60.00 t)
  - Remolacha[Tm]: 375.00 acres (Producción: 6000.00 t)

Compras realizadas (Toneladas):
  - Trigo[Tm] comprado: 0.00 t
  - Maíz[Tm] comprado: 180.00 t

Ventas realizadas (Toneladas):
  - Trigo[Tm] vendido: 0.00 t
  - Maíz[Tm] vendido: 0.00 t
  - Remolacha[Tm] (precio normal 26 €): 6000.00 t
  - Remolacha[Tm] (precio exceso 10 €): 0.00 t



<blockquote>

#### <font color='steelblue'> **Análisis soluciones**</font>





<blockquote>
    
#### <font color='steelblue'> **Conclusiones**:</font>



# <font color="steelblue">**Problema del transporte**</font>


![transporte](https://drive.google.com/thumbnail?id=1pgJvRucQWuqFmPEUKkcqEoe5nC2_uMLk&sz=h450)


  - $m$ plantas
  - $n$ clientes
  - $s_i$, recursos en la planta $i$, $i=1,\cdots ,m$
  - $d_j$, demanda del cliente $j$, $j=1,\cdots ,n$
  - $c_{ij}$, coste de transporte de la planta $i$ alcleinte $j$.

  - $x_{ij}$, variable de decisión que indica el  número de unidades a enviar desde la planta $i$ al cliente $j$.

\begin{align}
\min &\sum_{i}^m\sum_{j}^n c_{ij}x_{ij} \\
s.t.& \\
&\sum_{i=1}^m x_{ij}=d_j, \, \forall j=1,\cdots ,n\\
&\sum_{j=1}^n x_{ij}=s_i, \, \forall i=1,\cdots ,m\\
&x_{ij}\geq 0
\end{align}


In [26]:
#@title <font color="steelblue" size="+1"> Problema del transporte (CPLEX)
#Coste transporte
coste=[ [80, 215],
        [100,108],
        [102, 68],
        [0,    0]]
#recursos
s=[1000,1300,1200,200]
#demandas
d=[2300,1400]
#
m=range(len(s))
n=range(len(d))


In [24]:
def resolver_problema_transporte(
        coste,
        s,
        d,
        titulo="Problema de Transporte"):
    from docplex.mp.model import Model

    mdl = Model(name=titulo)

    # Índices basados en las dimensiones de las entradas
    m = range(len(s))
    n = range(len(d))

    # Variables de decisión: cantidad enviada de planta i a cliente j
    x = mdl.continuous_var_matrix(m, n, lb=0, name='x')

    # Función Objetivo: Minimizar costes totales de transporte
    coste_total = mdl.sum(coste[i][j] * x[i, j] for i in m for j in n)
    mdl.minimize(coste_total)

    # Restricciones de Demanda para cada cliente
    for j in n:
        mdl.add_constraint(mdl.sum(x[i, j] for i in m) == d[j], ctname=f'Demanda_{j}')

    # Restricciones de Capacidad (Suministro) para cada planta
    for i in m:
        mdl.add_constraint(mdl.sum(x[i, j] for j in n) == s[i], ctname=f'Suministro_{i}')

    # Resolver
    sol = mdl.solve()

    if sol:
        print(f"=== {titulo} ===")
        print(f"Estado de la solución: {mdl.get_solve_status()}")
        print(f"Coste Total de Transporte: {sol.objective_value:.2f} €")
        print("\nFlujos óptimos de transporte:")
        for i in m:
            for j in n:
                valor = x[i, j].solution_value
                if valor > 0:
                    print(f"  - Planta {i+1} -> Cliente {j+1}: {valor:.2f} unidades (Coste unitario: {coste[i][j]:.2f} €)")
        print("=" * (len(titulo) + 8) + "\n")
    else:
        print(f"No se encontró solución para: {titulo}")

    return mdl, sol

# Resolver el problema con los datos definidos en la celda anterior
mdl_transporte, sol_transporte = resolver_problema_transporte(coste, s, d)

=== Problema de Transporte ===
Estado de la solución: JobSolveStatus.OPTIMAL_SOLUTION
Coste Total de Transporte: 291600.00 €

Flujos óptimos de transporte:
  - Planta 1 -> Cliente 1: 1000.00 unidades (Coste unitario: 80.00 €)
  - Planta 2 -> Cliente 1: 1300.00 unidades (Coste unitario: 100.00 €)
  - Planta 3 -> Cliente 2: 1200.00 unidades (Coste unitario: 68.00 €)
  - Planta 4 -> Cliente 2: 200.00 unidades (Coste unitario: 0.00 €)



# <font color="steelblue">**Revenue Management problem**</font>

El Revenue Management (gestión de ingresos) es un enfoque de toma de decisiones que busca maximizar los ingresos esperados de una empresa mediante la asignación eficiente de recursos limitados frente a una demanda incierta y sensible al tiempo y al precio. Este problema surge típicamente en contextos donde la capacidad es fija o perecedera y no puede almacenarse para el futuro, como ocurre en aerolíneas, hoteles, transporte, telecomunicaciones y servicios de suscripción.

La idea central del Revenue Management consiste en vender el producto adecuado, al cliente adecuado, en el momento adecuado y al precio adecuado. Para lograrlo, las empresas deben decidir cómo asignar su capacidad entre distintos segmentos de clientes que presentan diferentes disposiciones a pagar, patrones de demanda y comportamientos de compra.

Desde un punto de vista formal, el problema de Revenue Management combina elementos de optimización, probabilidad y economía. Las decisiones deben tomarse de manera anticipada o secuencial, bajo incertidumbre, y suelen involucrar políticas de control como:

	- aceptación o rechazo de solicitudes de compra,
	- establecimiento de precios dinámicos,
	- protección de capacidad para clientes de mayor valor,
	- y asignación óptima de inventario a lo largo del tiempo.

Un rasgo fundamental del problema es la perecibilidad del recurso: una habitación de hotel o un asiento de avión que no se vende antes de la fecha de uso representa un ingreso perdido de manera irreversible. Esto obliga a balancear el riesgo de vender demasiado barato demasiado pronto contra el riesgo de dejar capacidad sin utilizar.

En la práctica, los modelos de Revenue Management se formulan como problemas de optimización estocástica, programación dinámica o aproximaciones deterministas, dependiendo del nivel de detalle y de la escala del sistema. Estos modelos permiten derivar políticas que, aunque no siempre sean óptimas en sentido estricto, resultan computacionalmente tratables y económicamente efectivas.

En resumen, el Revenue Management proporciona un marco cuantitativo para apoyar decisiones estratégicas y operativas en entornos donde la incertidumbre de la demanda y la limitación de recursos juegan un papel central, convirtiéndose en una de las aplicaciones más relevantes de la Investigación de Operaciones en la gestión moderna.

### <font color="steelblue">**Revenue Management problem**</font>

![revenue_management](https://drive.google.com/thumbnail?id=1bKHJK_fRlDz19VwOXrTjRWDf3vzheCnG&sz=h550)



  - $I$, conjunto de itinerarios
  - $J$, conjunto de vuelos.
  - $J_i \subset J$, conjunto de vuelos que forman el itinerario $i$, $i\in I$.   
  - $I_j \subset I$, conjunto de itinerarios que utilizan el vuelo $j$,  $j\in J$.   
  - $Q$, conjunto de clases
  - $r_i^q$, beneficio del itinerario $i$ para la calse $q$, $i\in I$, $q\in Q$
  - $c_j$, capacidad  del vuelo $j$, $j\in J$.
  - $D_{i}^{q}$, demanda del itinearario $i$ para la calse $q$,  $i\in I$, $q\in Q$.

  - $x_{i}^q$, variable de decisión que indica el  número de clientes que aceptaremos en cada una de las clases para el itinerario $i$,  $i\in I$, $q\in Q$.
  


\begin{align}
\max &\sum_{i\in I}\sum_{q\in Q} r_{i}^q x_{i}^q \\
s.t.& \\
&\sum_{q\in Q} \sum_{i\in I_j} x_{i}^q \leq c_j  , \, \forall j\in J\\
&0\leq x_{i}^q\leq D_{i}^{q}, \, \forall i\in I\\
\end{align}




In [12]:
#@title <font color="steelblue" size="+1"> Revenue Management (CPLEX)
#Itinerarios I /vuelos J
itinerarios=[[1,2],[1,3],[1,4],[2,1],[2,3],
             [2,4],[3,1],[3,2],[3,4],[4,1],
             [4,2],[4,3],
             [5,6],[6,5],[5,7],[7,5],[6,7],[7,6]]
vuelos=[[1,5],[5,1],[2,5],[5,2],[3,6],
        [6,3],[4,7],[7,4],
        [5,6],[6,5],[5,7],[7,5],[6,7],[7,6]]
J_i=[[1,4,-1],[1,9,6],[1,11,8],[3,2,-1],[3,9,6],
     [3,11,8],[5,10,2],[5,10,4],[5,13,8],[7,12,2],
     [7,12,4],[7,14,6],[9,-1,-1],[10,-1,-1],[11,-1,-1],[12,-1,-1],[13,-1,-1],[14,-1,-1]] # continuar
#clases
Clases=[1,2]
#demandas
D=[[100,5],[100,10],[50,10],[40,10],[50,20],[10,5],[20,5],[30,10],[40,10],[50,10],[40,10],[50,20],[20,5],[30,10],[40,10],[50,20],[40,10],[50,20]]
# Capacidades
C=[100,100,100,100,100,100,100,100,100,100,100,100,100,100]


#Beneficio
r=[[50,100],[20,500],[30,100],[40,100],[50,200],[50,100],[20,500],[30,100],[40,100],[50,200],[10,100],[50,200],[50,200],[50,200],[50,200],[50,200],[50,200],[50,200]]

I=range(len(itinerarios))
I=range(len(J_i))
K=range(3)
Q=range(len(Clases))
J=range(len(vuelos))
for j in J:
  C[j]=100


